In [1]:
# SPDX-FileCopyrightText: 2024-2026 Dan J. Bower <dbower@eaps.ethz.ch>
# SPDX-FileCopyrightText: 2024-2026 Paolo A. Sossi
#
# SPDX-License-Identifier: Apache-2.0

from pathlib import Path
from typing import Literal

from atmodeller import (
    ChemicalSpecies,
    EquilibriumModel,
    PurePhase,
    ThermodynamicState,
    console_logger,
)

# Log to the console. For more detail, use console_logger(logging.DEBUG)
logger = console_logger()

# Gas mixing

This notebook is available at `notebooks/examples/gas_mixing.ipynb` and is easiest to obtain by downloading the source code.

In gas mixing experiments, a predefined gaseous mixture is injected into a chamber and allowed to equilibrate thermodynamically under controlled temperature-pressure conditions. In this example, we consider a gas mixture that equilibrates in the presence of solid carbon (graphite).

In [2]:
# Whether to export the data to Excel and pickle files, and where
WRITE_OUTPUT = False
OUTPUT_DIRECTORY = Path("output") / "gas_mixing"

# There are a few different output formats
output_format: Literal["elements_species", "named_arrays"] = "named_arrays"

if WRITE_OUTPUT:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Define allowable gas species at equilibrium
H2_g = ChemicalSpecies.create_gas("H2")
N2_g = ChemicalSpecies.create_gas("N2")
CH4_g = ChemicalSpecies.create_gas("CH4")
CHN_g = ChemicalSpecies.create_gas("CHN")
H_g = ChemicalSpecies.create_gas("H")
gas_species = (H2_g, N2_g, CH4_g, CHN_g, H_g)

# Enforce graphite as a condensate (a pure phase in this model), so
# solve_for_stability=False
graphite = PurePhase.from_end_member("graphite", solve_for_stability=False)
condensate_phases = (graphite,)

# Set the thermodynamic state of the system
state = ThermodynamicState.from_species(
    gas_species, pressure=1, temperature=1773.15, melt_fraction=0, condensates=condensate_phases
)

# Define the mole fractions of input gases
mole_fractions = {"H2": 0.5, "N2": 0.5}

# Now define the model
model = EquilibriumModel.from_state(state, mass_constraints=mole_fractions, mass_units="moles")

# Solve with an automatic initial guess and default solver settings
output = model.solve_with_default()

# Optionally print solver stats to the logger
output.solver_stats_to_logger()

# Quick look at the solution
# output.quick_look()

# The abundance of C is not constrained, so the amount (number of moles and mass) of graphite is
# not solved for and is reported as NaN.

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output.to_excel(OUTPUT_DIRECTORY / "gas_mixing", output_format=output_format)

    # Write the data to a pickle file with dataframes
    output.to_pickle(OUTPUT_DIRECTORY / "gas_mixing", output_format=output_format)

[07:42:18 - atmodeller.phases              - INFO     ] - Creating PurePhase: ('C_graphite: ActivityCoefficient',)


[07:42:18 - atmodeller.phases              - INFO     ] - Creating GasPhase: ('H2_g: IdealGas', 'N2_g: IdealGas', 'CH4_g: IdealGas', 'CHN_g: IdealGas', 'H_g: IdealGas')


[07:42:18 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:42:18 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:42:18 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:42:18 - atmodeller.reactions           - INFO     ] - ReactionNetwork network = {0: '0.5 N2_g + 1.0 CH4_g = 1.5 H2_g + 1.0 CHN_g',
 1: '0.5 H2_g = 1.0 H_g',
 2: '1.0 CH4_g = 2.0 H2_g + 1.0 C_graphite'}


... (3 outputs left out)

[07:42:18 - atmodeller.classes             - INFO     ] - Initial guess: 1 model(s) from ElementPotentialInitializer, 0 model(s) from the given initial guess


[07:42:20 - atmodeller.output              - INFO     ] - Solve complete: 1 (100.00%) successful model(s)


[07:42:20 - atmodeller.output              - INFO     ] - Solve attempt 1: 1 (100.00%) model(s) solved


[07:42:20 - atmodeller.output              - INFO     ] - Solver steps (max) = 1
